## Importing libraries


In [ ]:

import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input


## Downloading the dataset

In [ ]:

import kagglehub

path = kagglehub.dataset_download(
    "rashikrahmanpritom/plant-disease-recognition-dataset"
)

print("Dataset root:", path)
print("Root folders:", os.listdir(path))
print("Inside Train:", os.listdir(os.path.join(path, "Train")))


In [ ]:
train_dir = os.path.join(path, "Train", "Train")
valid_dir = os.path.join(path, "Validation", "Validation")

print("Train classes:", os.listdir(train_dir))


## Data Generators

In [ ]:

IMG_SIZE = 224
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

print("Class indices:", train_generator.class_indices)
print("Number of classes:", train_generator.num_classes)


## Model Build

In [ ]:

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)

outputs = Dense(3, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=outputs)


In [ ]:

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:

history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=2
)


## saving the model

In [ ]:

model.save("plant_disease_efficientnet.h5")


In [ ]:
from tensorflow.keras.preprocessing import image

img_path = "plant_disease_input1.jpeg"

img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

pred = model.predict(img_array)
pred_class = np.argmax(pred, axis=1)[0]

class_labels = list(train_generator.class_indices.keys())
print("Predicted Disease:", class_labels[pred_class])


In [ ]:
model.summary